In [8]:
import os, csv, json
from datetime import datetime, timedelta, timezone

import psycopg
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool

load_dotenv()
DATABASE_URL = os.environ["DATABASE_URL"]
CURRENT_CUSTOMER_ID = 102

llm = ChatOpenAI(
    model=os.environ["OPENROUTER_MODEL"],
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0,
    max_tokens=1000,
)
print("Model configured. Current demo customer:", CURRENT_CUSTOMER_ID)

Model configured. Current demo customer: 102


In [9]:
create_tables_sql = """
CREATE TABLE IF NOT EXISTS customers (
    customer_id BIGINT PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS orders (
    order_id TEXT PRIMARY KEY,
    customer_id BIGINT NOT NULL REFERENCES customers(customer_id),
    restaurant TEXT NOT NULL,
    items TEXT NOT NULL,
    status TEXT NOT NULL,
    total_amount NUMERIC(10,2) NOT NULL,
    created_at TIMESTAMPTZ NOT NULL,
    promised_at TIMESTAMPTZ,
    delivered_at TIMESTAMPTZ
);
"""
with psycopg.connect(DATABASE_URL, connect_timeout=5) as conn:
    conn.execute(create_tables_sql)
    expected = {r[0] for r in conn.execute("SELECT column_name FROM information_schema.columns WHERE table_name='orders'").fetchall()}
    assert {"order_id", "customer_id", "status", "total_amount"}.issubset(expected), "Unexpected orders schema"


def read_csv(file_name):
    with open("data/" + file_name, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

customers = read_csv("customers.csv")
orders = read_csv("orders.csv")
now = datetime.now(timezone.utc)
with psycopg.connect(DATABASE_URL, connect_timeout=5) as conn:
    conn.execute("TRUNCATE orders, customers CASCADE")
    for row in customers:
        conn.execute("INSERT INTO customers(customer_id, name, email) VALUES (%s,%s,%s)",
                     (int(row["customer_id"]), row["name"], row["email"]))
    for row in orders:
        created = now - timedelta(hours=float(row["created_hours_ago"]))
        promised = created + timedelta(minutes=int(row["promised_minutes"]))
        delivered = None if not row["delivery_minutes"] else created + timedelta(minutes=int(row["delivery_minutes"]))
        conn.execute("""INSERT INTO orders(order_id, customer_id, restaurant, items, status, total_amount, created_at, promised_at, delivered_at)
                        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)""",
                     (row["order_id"], int(row["customer_id"]), row["restaurant"], row["items"], row["status"],
                      row["total_amount"], created, promised, delivered))
print("Loaded", len(customers), "customers and", len(orders), "orders")

Loaded 3 customers and 6 orders


##### Functions for Tools

In [10]:
def get_connection():
    return psycopg.connect(DATABASE_URL)

ORDER_COLUMNS = "order_id, restaurant, items, status, total_amount, created_at, promised_at, delivered_at"

def order_from_row(row):
    minutes_late = None
    if row[6] and row[7]:
        minutes_late = round((row[7] - row[6]).total_seconds() / 60)
    return {
        "order_id": row[0], "restaurant": row[1], "items": row[2], "status": row[3],
        "total_amount": float(row[4]), "created_at": str(row[5]),
        "promised_at": str(row[6]) if row[6] else None,
        "delivered_at": str(row[7]) if row[7] else None,
        "minutes_late": minutes_late,
    }

def query_latest_order(customer_id):
    with get_connection() as conn:
        return conn.execute(
            f"SELECT {ORDER_COLUMNS} FROM orders WHERE customer_id=%s ORDER BY created_at DESC LIMIT 1",
            (customer_id,),
        ).fetchone()

def query_order(customer_id, order_id):
    with get_connection() as conn:
        return conn.execute(
            f"SELECT {ORDER_COLUMNS} FROM orders WHERE order_id=%s AND customer_id=%s",
            (order_id, customer_id),
        ).fetchone()

In [11]:
@tool
def get_latest_order() -> dict:
    """Get the latest order belonging to the current classroom customer."""
    row = query_latest_order(CURRENT_CUSTOMER_ID)
    if row is None:
        return {"found": False, "message": "No orders were found for this customer."}
    return {"found": True, "order": order_from_row(row)}

@tool
def get_order_details(order_id: str) -> dict:
    """Get details of an order ID belonging to the current classroom customer."""
    row = query_order(CURRENT_CUSTOMER_ID, order_id)
    if row is None:
        return {"found": False, "message": "No matching order was found."}
    return {"found": True, "order": order_from_row(row)}

In [12]:
tools = [get_latest_order, get_order_details]
llm_with_tools = llm.bind_tools(tools)
for t in tools:
    print(t.name, t.description, t.args)

get_latest_order Get the latest order belonging to the current classroom customer. {}
get_order_details Get details of an order ID belonging to the current classroom customer. {'order_id': {'title': 'Order Id', 'type': 'string'}}


##### Implement Tool Calling Agent Manually

In [13]:
SYSTEM_PROMPT = (
    "You are a Zomato support assistant. Use tools to verify order details. "
    "Never invent order information. If a lookup returns no result, explain that you could not find it."
)


def run_basic_agent(question, max_rounds=5):
    messages = [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=question)]
    tool_map = {tool_obj.name: tool_obj for tool_obj in tools}

    for _ in range(max_rounds):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            return response.content

        for call in response.tool_calls:
            tool_obj = tool_map.get(call["name"])
            if tool_obj is None:
                result = {"error": "Unknown tool"}
            else:
                result = tool_obj.invoke(call["args"])

            messages.append(
                ToolMessage(
                    content=json.dumps(result, default=str),
                    tool_call_id=call["id"],
                )
            )

    return "I could not complete the request in this run. Please try again."


print(run_basic_agent("Where is my latest order?"))


Your latest order is from **Noodle Nest**. Here are the details:

- **Order ID:** ORD-2002
- **Items:** Hakka Noodles x1
- **Status:** PREPARING
- **Total Amount:** ₹220.00
- **Order Created At:** November 6, 2026, 05:48 AM (UTC)
- **Promised Delivery Time:** November 6, 2026, 06:23 AM (UTC)

Your order is currently being prepared and has not yet been delivered.
